In [ ]:
# 3x3 filter
# in_channels = 256

Conv2d(256, 256, kernel_size=3, stride=stride, padding=1, bias = False) # 이미지 크기 조절은 여기서 한 번만 진행
BatchNorm2d(256)

Conv2d(256, 256, kernel_size=3, stride=1, padding=1, bias=False) # image size 유지를 위해 padding=1(3x3)
BatchNorm2d(256)

In [ ]:
# downsample이 필요한 경우 -> x와 F(x) shape이 다른 경우
# 3x3 filter, in_channels = 64, 1st conv : stride = 2
# x = [64, 56, 56], F[x] = [128, 28, 28]
# channel은 64 -> 128로, size는 56 -> 28로 

# down_sample 수행
nn.Conv2d(64, 128, kernel_size=1, stride=2, bias=False), # 64 = input_channel, 128 = output_channel
nn.BatchNorm2d(128), 

In [ ]:
# Basic Block 정의 
class BasicBlock(nn.Module):
    expansion=1 # 블록을 거치고 나서 채널이 몇 배로 늘어나는지

    def __init__(self, in_channels, out_channels, stride=1, downsample=False):
        super.__init__()
        # 첫 번째 Conv 블록 
        # 여기에서 stride=2면 이미지 크기가 절반으로 줄어듦 
        # 크기 조절은 여기서 한 번만 진행
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        # 두 번째 Conv 블록
        # stride=1 고정이라 크기 유지 
        # 입력도 out_channels, 출력도 out_channels (conv1 출력을 받기 때문)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias = False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)

        # downsample 처리
        if downsample:
            conv = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False)
            bn = nn.BatchNorm2d(out_channels)

            # nn.Sequential은 여러 레이러를 순서대로 묶어주는 컨테이너이다. 
            # x -> conv -> bn -> 출력 
            downsample = nn.Sequential(conv, bn) # skip connection 경로

        else:
            downsample = None

        self.downsample = downsample


In [ ]:
# Bottleneck
class Bottleneck(nn.Module):
    expansion = 4 # ResNet에서 Bottleneck block을 정의하기 위한 하이퍼파라미터이다. (채널 수 4배 : 64 -> 256)

    def __init__(self, in_channels, out_channels, stride=1, downsample=False):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, bias = False)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels) # 이미지 크기 조절은 여기에서만 

        self.conv3 = nn.Conv2d(out_channels, self.expansion*out_channels, kernel_size=1, stride=1, bias=False)
        self.bn3 = nn.BatchNorm2d(self.expansion*out_channels)

        self.relu = nn.ReLU(inplace=True)

        if downsample:
            conv = nn.Conv2d(in_channels, self.expansion*out_channels, kernel_size=1, stride=stride, bias=False)
            bn = nn.BatchNorm2d(self.expansion*out_channels)
            downsample = nn.Sequential(conv, bn)

        else:
            downsample:None
        
        self.downsample = downsample

    def forward(self, x):
        i=x
        x=self.conv1(x)
        x=self.bn1(x)
        x=self.relu(x)

        x=self.conv2(x)
        x=self.bn2(x)
        x=self.relu(x)


        x=self.conv3(x)
        x=self.bn3(x)

        if self.downsample is not None:
            i = self.downsample(i)

        x += i
        x=self.relu(x)
        return x


In [ ]:
# Basic block & Bottleneck block Implementation

def resnet18():
   return ResNet(BasicBlock, [2,2,2,2])
 
def resnet34():
    return ResNet(BasicBlock, [3,4,6,3])

def resnet50():
    return ResNet(BottleNeck, [3,4,6,3])

def resnet101():
    return ResNet(BottleNeck, [3,4,23,3])

def resnet152():
    return ResNet(BottleNeck, [3,8,36,3])

class ResNet(nn.Module):
    # 초기 설정
    def __init__(self, block, num_block, num_classes=10, init_weights=True):
        super().__init__()
        self.in_channels=64 # 앞으로 블록을 만들 때 입력 채널 추적용
        # 첫 번째 Conv (7x7) - 이미지의 해상도를 줄이면서 초기 특징을 뽑는 부분
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False) # 3채널(RGB)->64채널, 크기 절반
            nn.BatchNorm2d(64)
            nn.ReLU()
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1) # 크기 또 절반(크기 1/4)
        )
        # _make_layer를 옽해 반복되는 잔차 블록들을 생성한다. 
        # 스테이즈가 넘어갈 때마다 출력 채널이 64 -> 128 -> 256 -> 512로 늘어난다. 
        self.conv2_x=self._make_layer(block, 64, num_block[0],1) # 1st layer
        self.conv3_x=self._make_layer(block, 128, num_block[1], 2) # 2nd layer
        self.conv4_x=self._make_layer(block, 256, num_block[2], 2) # 3rd layer
        self.conv5_x = self._make_layer(block, 512, num_block[3], 2) # 4th layer

        # 마지막 레이어
        self.avg_pool = nn.AdaptiveAvgPool2d((1,1)) # 어떤 크기든 1x1로 압축
        # 512*block.expansion : 마지막 스테이지의 출력 채널 수 
        self.fc=nn.Linear(512*block.expansion, num_classes) # fc layer (최종 분류)

        if init_weights:
            self._initialize_weights()

    # 핵심 로직 
    # 블록을 list에 담아 nn.Sequential로 묶어준다. 
    def _make_layer(self, block, out_channels, num_blocks, stride):
        # 첫 번째 블록만 입력받은 stride를 사용하여 이미지 크기를 줄인다. 
        # 나머지 num_block -1개의 블록은 stride=1을 사용하여 크기를 유지한다. 
        strides=[stride] + [1]*(num_blocks-1) 
        layers=[]
        for stride in strides: 
            layers.append(block(self.in_channels, out_channels, stride))
            self.in_channels = out_channels*block.expansion # 현재 채널 상태 갱신

        return nn.Sequential(*layers)

    def forward(self, x):
        output =self.conv1(x)
        output = self.conv2_x(output)
        x=self.conv3_x(output)
        x=self.conv4_x(x)
        x=self.conv5_x(x)
        x=self.avg_pool(x)
        x=x.view(x.size(0),-1)
        x=self.fc(x)
        return x
    
    # ResNet 모델의 가중치 초기화
    def _initialize_weights(self):
        for m in self.modules(): # 모델 안에 있는 모든 layer/블록 순회
            if isinstance(m, nn.Conv2d): # m이 Conv2d인지 확인
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: # Conv Layer에 bias가 있으면
                    nn.init.constant_(m.bias, 0) # bias=False로 초기화
            elif isinstance(m, nn.BatchNorm2d): # 현재 순회 중인 모듈 m이 BatchNorm2d인지 확인
                nn.init.constant_(m.weight,1) # BatchNorm의 scale =1
                nn.init.constant_(m.bias,0) # BatchNorm의 bias =0
            elif isinstance(m, nn.Linear): # 현재 순회중인 모듈 m이 Linear Layer인지 확안
                nn.init.normal_(m.weight, 0, 0.01) # Linear weight를 정규분포(normal)로 초기화
                nn.init.constant_(m.bias, 0) # bias=0으로 초기화

class BasicBlock(nn.Module):
    expansion=1
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.residual_function = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels*BasicBlock.expansion, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_channels*BasicBlock.expansion)
        )
        self.shortcut = nn.Sequential()
        self.relu = nn.ReLU()

        # 이미지 크기가 다르거나, 채널 개수가 다르면,
        if stride !=1 or in_channels !=BasicBlock.expansion*out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels*BasicBlock.expansion, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels*BasicBlock.expansion)
            )
    def forward(self,x):
        x=self.residual_function(x) + self.shortcut(x)
        x=self.relu(x)
        return x
    
class BottleNeck(nn.Module):
    expansion=4
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.residual_function= nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1,stride=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels*BottleNeck.expansion, kernel_size=1, stride=1, bias=False),
            nn.BatchNorm2d(out_channels*BottleNeck.expansion)
        )
        self.shortcut = nn.Sequential()
        self.relu = nn.ReLU()

        if stride!=1 or in_channels != out_channels*BottleNeck.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels*BottleNeck.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels*BottleNeck.expansion)
            )


    def forward(self, x):
        x=self.residual_function(x) + self.shortcut(x)
        x=self.relu(x)
        return x

